In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

In [2]:
df = pd.read_csv("cleaned_transactions.csv")
df.shape

(6362620, 14)

## Prepare the features
Fraud only happens in TRANSFER and CASH_OUT, so scoping the model to just those two types. Not using `fraud_risk_flag` as input that would be cheating since it's built from the answer itself.

In [3]:
model_df = df[df['payment_type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
model_df['is_transfer'] = (model_df['payment_type'] == 'TRANSFER').astype(int)
features = ['is_transfer', 'money_amount', 'sender_balance_before',
            'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after']
X = model_df[features]
y = model_df['is_fraud']
X.shape, y.sum(), y.mean()

((2770409, 6), 8213, 0.002964544224336551)

Fraud is only ~0.3% of this subset rare enough that accuracy alone would be a useless metric (a model predicting "not fraud" every time would already look 99.7% accurate).

## Split into train and test sets
Stratified split  keeps the same fraud percentage in both train and test, important since fraud is rare (~1.5% of this subset).

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.sum(), y_test.sum()

((2216327, 6), (554082, 6), 6570, 1643)

## Model 1: Logistic Regression
Simple baseline model fast, interpretable, good starting point.

In [5]:
log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)
print(classification_report(y_test, y_pred_log, digits=4))

              precision    recall  f1-score   support

           0     0.9996    0.9351    0.9663    552439
           1     0.0388    0.8807    0.0743      1643

    accuracy                         0.9349    554082
   macro avg     0.5192    0.9079    0.5203    554082
weighted avg     0.9968    0.9349    0.9636    554082



**Note:** `class_weight='balanced'` makes the model prioritize catching fraud over avoiding false alarms without it, accuracy would look great while catching zero fraud.

**Result:** recall 0.88 (catches 88% of real fraud), precision 0.039 (lots of false alarms, but few real frauds missed).

## Model 2: Decision Tree
Second baseline captures non-linear patterns logistic regression might miss.

In [6]:
tree_model = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)
print(classification_report(y_test, y_pred_tree, digits=4))

              precision    recall  f1-score   support

           0     1.0000    0.9446    0.9715    552439
           1     0.0505    0.9903    0.0961      1643

    accuracy                         0.9448    554082
   macro avg     0.5252    0.9675    0.5338    554082
weighted avg     0.9972    0.9448    0.9689    554082



**Why precision is low:** `class_weight='balanced'` favors catching fraud over avoiding false alarms  normal trade-off for a first-pass model, not a mistake.

**Result:** Decision Tree beats Logistic Regression on recall (0.99 vs 0.88) catches almost every fraud case, slightly more false alarms.

## Compare both models directly
Side by side on precision and recall for the fraud class specifically.

In [7]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Precision (fraud)": [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_tree)
    ],
    "Recall (fraud)": [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_tree)
    ]
})
results

,Model,Precision (fraud),Recall (fraud)
0,Logistic Regression,0.038768,0.880706
1,Decision Tree,0.050519,0.990262


## Save risk scores back onto the dataset
Adds a `ml_fraud_probability` column — this is what would feed the Power BI risk-scoring page.

In [8]:
model_df['ml_fraud_probability'] = log_model.predict_proba(X)[:, 1]
model_df[['payment_type', 'money_amount', 'is_fraud', 'ml_fraud_probability']].sort_values(
    'ml_fraud_probability', ascending=False
).head(10)

,payment_type,money_amount,is_fraud,ml_fraud_probability
2041058,TRANSFER,1644749.94,1,1.0
6009595,TRANSFER,10000000.00,1,1.0
6347723,TRANSFER,5175876.53,1,1.0
6008702,TRANSFER,1648547.69,1,1.0
6347724,CASH_OUT,5175876.53,1,1.0
4202855,TRANSFER,4164236.30,0,1.0
6009371,TRANSFER,3653274.35,1,1.0
6009372,CASH_OUT,3653274.35,1,1.0
6009393,TRANSFER,3947541.37,1,1.0
6009394,CASH_OUT,3947541.37,1,1.0


## Compare ML model vs the fraud_risk_flag rule vs the old system
Same recall comparison as before, now including the ML model — shows how far the project has come from the original 0.19% detection rate.

In [9]:
compare = pd.DataFrame({
    "Method": ["Old system (system_caught_it)", "fraud_risk_flag (rule-based)", "Logistic Regression (ML)"],
    "Recall": [
        recall_score(model_df['is_fraud'], model_df['system_caught_it']),
        recall_score(model_df['is_fraud'], model_df['fraud_risk_flag']),
        recall_score(y_test, y_pred_log)
    ]
})
compare

,Method,Recall
0,Old system (system_caught_it),0.001948
1,fraud_risk_flag (rule-based),0.994521
2,Logistic Regression (ML),0.880706


## Key takeaways
- Model scoped to TRANSFER/CASH_OUT only — fraud never occurs elsewhere.
- `fraud_risk_flag` (no ML needed) is compared directly against the ML model — simple rules are often easier to explain to business people.
- `ml_fraud_probability` is a 0-1 score, not flat yes/no — lets the business set its own risk threshold.
- Both models trade precision for recall on purpose — catch more fraud, accept more false alarms.
- Baseline model, not production-grade — good enough to show the workflow and feed the dashboard.